In [2]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

freq_bb = np.fft.rfftfreq(N, 1/fs)

for ax, mu, color, title in zip(axes, [0.5, 1.5], ['green', 'red'],
                                  ['μ = 0.5 — detector recupera m(t)', 
                                   'μ = 1.5 — detector produz |1+μm(t)|']):
    envelope = 1 + mu * m
    detector_output = np.abs(envelope)
    # Remover DC para ver só as componentes AC
    detector_ac = detector_output - np.mean(detector_output)
    
    D = np.fft.rfft(detector_ac) / N
    D_mag = 2 * np.abs(D)
    
    ax.plot(freq_bb/1e3, D_mag, color, lw=1.2)
    ax.set_ylabel('|D(f)|')
    ax.set_title(title)
    ax.set_xlim([0, 6])
    
    # Marcar fundamental
    ax.axvline(fm/1e3, color='blue', ls=':', lw=1.5, alpha=0.5)
    ax.annotate('$f_m$', xy=(fm/1e3, D_mag.max()*0.85), fontsize=11, 
                ha='center', color='blue')
    
    # Marcar harmônicas
    if mu > 1:
        for n in [2, 3, 4]:
            f_h = n * fm
            ax.axvline(f_h/1e3, color='orange', ls='--', lw=1, alpha=0.7)
            ax.annotate(f'${n}f_m$', xy=(f_h/1e3, D_mag.max()*0.7), fontsize=10,
                        ha='center', color='orange')

axes[-1].set_xlabel('Frequência (kHz)')
fig.suptitle('Espectro da Saída do Detector de Envelope (banda base)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("="*60)
print("CONCLUSÃO:")
print(f"  μ = 0.5 → espectro limpo, apenas fm = {fm} Hz")
print(f"  μ = 1.5 → harmônicas espúrias em {2*fm}, {3*fm}, {4*fm} Hz ...")
print(f"           → sinal recuperado DISTORCIDO!")
print("="*60)

NameError: name 'plt' is not defined

## 4. Espectro da saída do detector de envelope

Aqui fica mais evidente: o espectro do sinal recuperado com $\mu = 0.5$ tem só o tom em $f_m$. Com $\mu = 1.5$, aparecem harmônicas em $2f_m$, $3f_m$, ... que **não existiam** no sinal original.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

freq = np.fft.rfftfreq(N, 1/fs)

for ax, mu, color, title in zip(axes, [0.5, 1.5], ['blue', 'red'],
                                  ['μ = 0.5 (normal)', 'μ = 1.5 (supermodulação)']):
    envelope = Ac * (1 + mu * m)
    s_am = envelope * np.cos(2*np.pi*fc*t)
    
    # Espectro
    S = np.fft.rfft(s_am) / N
    S_mag = 2 * np.abs(S)
    
    ax.plot(freq/1e3, S_mag, color, lw=1.2)
    ax.set_ylabel('|S(f)|')
    ax.set_title(title)
    ax.set_xlim([7, 14])
    
    # Marcar portadora e bandas laterais
    ax.axvline(fc/1e3, color='gray', ls=':', lw=1, alpha=0.5)
    ax.annotate('$f_c$', xy=(fc/1e3, ax.get_ylim()[1]*0.9 if mu < 1 else 0.9),
                fontsize=10, ha='center', color='gray')
    
    # Marcar harmônicas espúrias
    if mu > 1:
        for n in [2, 3]:
            for sign in [-1, 1]:
                f_harm = fc + sign * n * fm
                ax.axvline(f_harm/1e3, color='orange', ls='--', lw=1, alpha=0.7)
        ax.annotate('harmônicas\nespúrias', xy=(12.2, 0.05), fontsize=10,
                    color='orange', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='orange'),
                    xytext=(13, 0.15))

axes[-1].set_xlabel('Frequência (kHz)')
fig.suptitle('Espectro do Sinal AM Transmitido', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Comparação espectral: harmônicas espúrias

O espectro do sinal normal tem apenas a portadora ($f_c$) e as bandas laterais ($f_c \pm f_m$).

Com supermodulação, a operação $|1+\mu\,m(t)|$ introduz harmônicas ($2f_m, 3f_m, \ldots$) que aparecem em $f_c \pm 2f_m$, $f_c \pm 3f_m$, etc.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, mu, color in zip(axes, [0.5, 1.5], ['green', 'red']):
    envelope = 1 + mu * m
    detector_output = np.abs(envelope)  # |1 + μ m(t)|
    
    ax.plot(t*1e3, envelope, 'b--', lw=1.5, label='$1 + \\mu\\,m(t)$ (ideal)')
    ax.plot(t*1e3, detector_output, color, lw=2, label='$|1 + \\mu\\,m(t)|$ (detector)')
    ax.axhline(0, color='k', lw=0.5)
    
    if mu > 1:
        neg = envelope < 0
        ax.fill_between(t*1e3, envelope, detector_output, where=neg,
                        alpha=0.3, color='orange', label='Distorção')
    
    ax.set_ylabel('Amplitude')
    ax.set_title(f'μ = {mu}')
    ax.legend(loc='upper right')
    ax.set_xlim([0, 3])

axes[-1].set_xlabel('Tempo (ms)')
fig.suptitle('Saída do Detector de Envelope', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Saída do detector de envelope

O detector de envelope extrai $|1 + \mu\,m(t)|$. Quando $\mu \leq 1$, recupera $m(t)$ perfeitamente. Quando $\mu > 1$, as partes negativas "rebatemp" para cima → **distorção**.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, mu, title in zip(axes, [0.5, 1.5], 
                           ['μ = 0.5 (normal)', 'μ = 1.5 (supermodulação)']):
    envelope = Ac * (1 + mu * m)
    s_am = envelope * np.cos(2*np.pi*fc*t)
    
    ax.plot(t*1e3, s_am, 'b', lw=0.6, alpha=0.7, label='$s(t)$')
    ax.plot(t*1e3, envelope, 'r-', lw=1.5, label='Envelope $1+\\mu\\,m(t)$')
    ax.plot(t*1e3, -envelope, 'r-', lw=1.5)
    ax.axhline(0, color='k', lw=0.5)
    
    if mu > 1:
        neg = envelope < 0
        ax.fill_between(t*1e3, ax.get_ylim()[0]*np.ones_like(t), 
                        ax.get_ylim()[1]*np.ones_like(t),
                        where=neg, alpha=0.15, color='red', label='Envelope negativo')
    
    ax.set_ylabel('Amplitude')
    ax.set_title(title)
    ax.legend(loc='upper right')
    ax.set_xlim([0, 3])

axes[-1].set_xlabel('Tempo (ms)')
plt.tight_layout()
plt.show()

## 1. Comparação no domínio do tempo: $\mu = 0.5$ vs $\mu = 1.5$

Com $\mu = 1.5$, o envelope $1 + \mu\,m(t)$ fica negativo → a portadora **inverte a fase** (salto de 180°).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

# Parâmetros
fm = 1000       # Frequência da mensagem (Hz)
fc = 10000      # Frequência da portadora (Hz)
Ac = 1.0        # Amplitude da portadora
fs = 200000     # Taxa de amostragem (alta para boa resolução espectral)
T = 0.02        # Duração (20 ms — vários ciclos para FFT limpa)

t = np.arange(0, T, 1/fs)
N = len(t)
m = np.cos(2*np.pi*fm*t)  # Mensagem normalizada (tom puro)

# Supermodulação AM: Visualização da Inversão de Fase e Distorção Espectral

Este notebook mostra o que acontece quando $\mu > 1$ no AM convencional:
1. **Sinal no tempo** — envelope negativo e inversão de fase
2. **Saída do detector de envelope** — distorção por "rebatimento"
3. **Espectro** — harmônicas espúrias fora da banda $2W$